# AdventureWorks 2025 retail analytics dashboard

The actual dashboard is in a separate html file. The following charts are duplicated in order to provide additional text analysis with insights.

## Setup

In [34]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.offline as plto
from pathlib import Path

from adventureworks_data import build_payload

plto.init_notebook_mode(connected=True)

payload = build_payload(Path("data") / "part3")
sales_rows = pd.DataFrame(payload["sales"])
client_rows = pd.DataFrame(payload["clients"])
stock_rows = pd.DataFrame(payload["stock"])
demand_rows = pd.DataFrame(payload["demand"])
meta = payload["meta"]

sales_rows["month"] = pd.to_datetime(sales_rows["month"])
client_rows["firstOrderYear"] = pd.to_numeric(client_rows["firstOrderYear"], errors="coerce")
client_rows["age"] = pd.to_numeric(client_rows["age"], errors="coerce")
demand_rows["daysOfCover"] = pd.to_numeric(demand_rows["daysOfCover"], errors="coerce")

incomplete_month = sales_rows["month"].max()
trend_sales = sales_rows[sales_rows["month"] != incomplete_month].copy()

tier_order = ["Low priority", "Active core", "At-risk value", "Priority active"]
tier_colors = {
    "Low priority": "#98a2b3",
    "Active core": "#53b1fd",
    "At-risk value": "#f79009",
    "Priority active": "#12b76a",
}
category_colors = {
    "Bikes": "#1769aa",
    "Components": "#7a5af8",
    "Clothing": "#2e7d32",
    "Accessories": "#f79009",
}

plotly_template = "plotly_white"


def money(value):
    return f"${value:,.0f}"


def number(value):
    return f"{value:,.0f}"


def apply_dashboard_layout(fig, title, height=460, showlegend=False):
    fig.update_layout(
        title=title,
        template=plotly_template,
        height=height,
        showlegend=showlegend,
        margin=dict(l=64, r=48, t=72, b=64),
        hoverlabel=dict(align="left"),
        font=dict(family="Segoe UI, Arial, sans-serif", color="#172033", size=12),
    )
    return fig


def revenue_bar(data, x, y, title, orientation="v", color="#1769aa", height=430):
    fig = go.Figure(
        go.Bar(
            x=data[x],
            y=data[y],
            orientation=orientation,
            marker_color=color,
            hovertemplate=(
                "%{y}<br>$%{x:,.0f}<extra></extra>"
                if orientation == "h"
                else "%{x}<br>$%{y:,.0f}<extra></extra>"
            ),
        )
    )
    apply_dashboard_layout(fig, title, height=height)
    if orientation == "h":
        fig.update_xaxes(tickprefix="$", separatethousands=True)
        fig.update_yaxes(automargin=True)
    else:
        fig.update_yaxes(tickprefix="$", separatethousands=True)
    return fig

print(
    f"Dashboard data loaded: {len(sales_rows):,} sales lines, "
    f"{len(client_rows):,} clients, {len(stock_rows):,} stock rows."
)
print(f"Trend charts exclude the incomplete final month: {incomplete_month:%B %Y}.")

Dashboard data loaded: 60,398 sales lines, 18,484 clients, 1,141 stock rows.
Trend charts exclude the incomplete final month: January 2014.


## Overview

In [35]:
# Monthly revenue and profit
monthly = (
    trend_sales.groupby("month", as_index=False)
    .agg(revenue=("revenue", "sum"), profit=("profit", "sum"), customers=("customer", "nunique"))
    .sort_values("month")
)
fig = go.Figure()
fig.add_trace(go.Scatter(x=monthly["month"], y=monthly["revenue"], mode="lines+markers", name="Revenue", line=dict(color="#1769aa", width=3), hovertemplate="%{x|%b %Y}<br>Revenue: $%{y:,.0f}<extra></extra>"))
fig.add_trace(go.Scatter(x=monthly["month"], y=monthly["profit"], mode="lines+markers", name="Profit", line=dict(color="#2e7d32", width=3), hovertemplate="%{x|%b %Y}<br>Profit: $%{y:,.0f}<extra></extra>"))
fig.add_trace(go.Scatter(x=monthly["month"], y=monthly["customers"], mode="lines", name="Buying customers", yaxis="y2", line=dict(color="#7a5af8", width=2, dash="dot"), hovertemplate="%{x|%b %Y}<br>Buying customers: %{y:,.0f}<extra></extra>"))
apply_dashboard_layout(fig, "Monthly revenue and profit", height=500, showlegend=True)
fig.update_yaxes(title_text="Revenue / profit", tickprefix="$", separatethousands=True)
fig.update_layout(yaxis2=dict(title="Buying customers", overlaying="y", side="right", rangemode="tozero", showgrid=False), hovermode="x unified")
fig.show()

# Revenue by category
category_revenue = sales_rows.groupby("category", as_index=False)["revenue"].sum().sort_values("revenue", ascending=False).head(8)
fig = revenue_bar(category_revenue, "category", "revenue", "Revenue by category", color="#1769aa")
fig.show()

# Revenue by country
country_revenue = sales_rows.groupby("country", as_index=False)["revenue"].sum().sort_values("revenue", ascending=False).head(8)
fig = revenue_bar(country_revenue, "country", "revenue", "Revenue by country", color="#53b1fd")
fig.show()

**Idea behind the design choice:** The Overview is meant to answer "how is the business doing?" at a glance before going into details. The monthly line chart puts revenue and profit on the same time axis so the gap between them (the margin) can be read directly and adds the number of buying customers on a secondary axis to tell whether revenue moves because of more customers or because the customers started spending more. The two bar charts then split that same total revenue by category and by country, so we immediately see where the money comes from.

**Possible insights:** Revenue and profit rise together and track each other closely, so growth stays profitable. Bikes dominate the category breakdown by a wide margin, which means the company is effectively a bike business with accessories attached and several of countries account for most of the sales.

**Usage of interactions:** Unified hover on the time chart gives exact revenue, profit and customer values for each month, hovering the bars shows precise totals, and the legend entries can be toggled to isolate a single series.

## Clients

In [36]:
# Revenue vs activity tiers
fig = go.Figure()
for tier in tier_order:
    rows = client_rows[client_rows["valueTier"] == tier]
    fig.add_trace(
        go.Scatter(
            x=rows["activityScore"],
            y=rows["revenue"],
            mode="markers",
            name=tier,
            marker=dict(size=6, opacity=0.58, color=tier_colors[tier]),
            customdata=np.stack([rows["country"], rows["occupation"], rows["incomeGroup"], rows["orders"], rows["daysSinceLastOrder"], rows["avgOrderValue"], rows["categories"]], axis=-1),
            hovertemplate=(
                f"<b>{tier}</b><br>Revenue: $%{{y:,.0f}}<br>Activity score: %{{x:.1f}}<br>"
                "Country: %{customdata[0]}<br>Occupation: %{customdata[1]}<br>Income group: %{customdata[2]}<br>"
                "Orders: %{customdata[3]:,.0f}<br>Days since last order: %{customdata[4]:,.0f}<br>"
                "Avg order value: $%{customdata[5]:,.0f}<br>Categories: %{customdata[6]:,.0f}<extra></extra>"
            ),
        )
    )
apply_dashboard_layout(fig, "Revenue vs activity tiers", height=560, showlegend=True)
fig.add_hline(y=meta["clientThresholds"]["revenueHigh"], line_dash="dash", line_color="#667085", annotation_text="High revenue cutoff", annotation_position="top left")
fig.add_vline(x=meta["clientThresholds"]["activityHigh"], line_dash="dash", line_color="#667085", annotation_text="High activity cutoff", annotation_position="top right")
fig.update_xaxes(title_text="Customer activity score", range=[0, 100])
fig.update_yaxes(title_text="Total customer revenue", tickprefix="$", separatethousands=True)
fig.show()

# Revenue distribution by value tier
fig = go.Figure()
for tier in tier_order:
    rows = client_rows[client_rows["valueTier"] == tier]
    fig.add_trace(go.Box(y=rows["revenue"], name=tier, marker_color=tier_colors[tier], boxpoints=False, hovertemplate=f"<b>{tier}</b><br>Revenue: $%{{y:,.0f}}<extra></extra>"))
apply_dashboard_layout(fig, "Revenue distribution by value tier", height=500)
fig.update_yaxes(title_text="Customer revenue", tickprefix="$", separatethousands=True)
fig.update_xaxes(title_text="Customer value tier")
fig.show()

# Revenue share by tier
tier_revenue = client_rows.groupby("valueTier", as_index=False)["revenue"].sum()
tier_revenue["valueTier"] = pd.Categorical(tier_revenue["valueTier"], categories=tier_order, ordered=True)
tier_revenue = tier_revenue.sort_values("valueTier")
fig = go.Figure(
    go.Pie(
        labels=tier_revenue["valueTier"],
        values=tier_revenue["revenue"],
        hole=0.55,
        sort=False,
        marker=dict(colors=[tier_colors[t] for t in tier_revenue["valueTier"]]),
        textinfo="label+percent",
        hovertemplate="%{label}<br>Revenue: $%{value:,.0f}<br>Share: %{percent}<extra></extra>",
    )
)
apply_dashboard_layout(fig, "Revenue share by tier", height=460)
fig.show()

# Customer revenue concentration
revenues = client_rows["revenue"].dropna()
revenues = revenues[revenues > 0].sort_values(ascending=False).reset_index(drop=True)
cumulative_customers = (np.arange(len(revenues)) + 1) / len(revenues) * 100
cumulative_revenue = revenues.cumsum() / revenues.sum() * 100
top20_index = max(0, min(len(revenues) - 1, round(0.2 * len(revenues)) - 1))
top20_share = cumulative_revenue.iloc[top20_index]
fig = go.Figure()
fig.add_trace(go.Scatter(x=cumulative_customers, y=cumulative_revenue, mode="lines", name="Cumulative revenue", line=dict(color="#1769aa", width=3), hovertemplate="Top %{x:.0f}% of customers<br>%{y:.1f}% of revenue<extra></extra>"))
fig.add_trace(go.Scatter(x=[0, 100], y=[0, 100], mode="lines", name="Equal contribution", line=dict(color="#cfd6e2", width=1, dash="dot"), hoverinfo="skip"))
apply_dashboard_layout(fig, "Customer revenue concentration (Pareto)", height=500, showlegend=True)
fig.add_vline(x=20, line_dash="dash", line_color="#ba1a1a", annotation_text=f"Top 20% -> {top20_share:.0f}%", annotation_position="top left")
fig.update_xaxes(title_text="Cumulative share of customers", ticksuffix="%", range=[0, 100])
fig.update_yaxes(title_text="Cumulative share of revenue", ticksuffix="%", range=[0, 100])
fig.show()

# Value-tier composition by occupation
occupation_counts = pd.crosstab(client_rows["occupation"], client_rows["valueTier"]).reindex(columns=tier_order, fill_value=0)
occupation_share = occupation_counts.div(occupation_counts.sum(axis=1), axis=0).fillna(0)
occupation_revenue = client_rows.pivot_table(index="occupation", columns="valueTier", values="revenue", aggfunc="sum", fill_value=0).reindex(columns=tier_order, fill_value=0)
custom = np.dstack([occupation_counts[tier_order].values, occupation_revenue[tier_order].values])
fig = go.Figure(
    go.Heatmap(
        z=occupation_share[tier_order].values,
        x=tier_order,
        y=occupation_share.index,
        colorscale="Blues",
        reversescale=True,
        zmin=0,
        zmax=1,
        texttemplate="%{z:.0%}",
        textfont=dict(size=11, color="#1d1d1f"),
        customdata=custom,
        colorbar=dict(title=dict(text="Share of<br>occupation"), tickformat=".0%"),
        hovertemplate="Occupation: %{y}<br>Tier: %{x}<br>Share of occupation: %{z:.1%}<br>Customers: %{customdata[0]:,.0f}<br>Revenue: $%{customdata[1]:,.0f}<extra></extra>",
    )
)
apply_dashboard_layout(fig, "Value-tier composition by occupation", height=560)
fig.update_xaxes(title_text="Customer value tier")
fig.update_yaxes(title_text="Occupation", automargin=True)
fig.show()

# Top demographic segments
segments = (
    client_rows.groupby(["incomeGroup", "occupation"], as_index=False)
    .agg(revenue=("revenue", "sum"), customers=("customer", "nunique"))
    .sort_values("revenue", ascending=False)
)
segments["segment"] = segments["incomeGroup"] + " | " + segments["occupation"]
segments["avgRevenue"] = segments["revenue"] / segments["customers"]
top_segments = segments.head(12).sort_values("revenue")
fig = go.Figure(go.Bar(x=top_segments["revenue"], y=top_segments["segment"], orientation="h", marker_color="#2e7d32", customdata=np.stack([top_segments["customers"], top_segments["avgRevenue"]], axis=-1), hovertemplate="%{y}<br>Revenue: $%{x:,.0f}<br>Customers: %{customdata[0]:,.0f}<br>Avg/customer: $%{customdata[1]:,.0f}<extra></extra>"))
apply_dashboard_layout(fig, "Top demographic segments", height=560)
fig.update_xaxes(tickprefix="$", separatethousands=True)
fig.update_yaxes(automargin=True)
fig.show()

# What each value tier buys
mix_categories = ["Bikes", "Components", "Clothing", "Accessories"]
customer_tier = client_rows.set_index("customer")["valueTier"]
tier_mix_sales = sales_rows[sales_rows["category"].isin(mix_categories)].copy()
tier_mix_sales["valueTier"] = tier_mix_sales["customer"].map(customer_tier)
tier_mix_sales = tier_mix_sales.dropna(subset=["valueTier"])
tier_cat_lines = (
    tier_mix_sales.groupby(["valueTier", "category"]).size()
    .unstack(fill_value=0)
    .reindex(index=tier_order, columns=mix_categories, fill_value=0)
)
tier_cat_share = tier_cat_lines.div(tier_cat_lines.sum(axis=1), axis=0).fillna(0)
mix_tiers = [tier for tier in tier_order if tier_cat_lines.loc[tier].sum() > 0]
fig = go.Figure()
for category in mix_categories:
    fig.add_trace(
        go.Bar(
            y=mix_tiers,
            x=tier_cat_share.loc[mix_tiers, category],
            name=category,
            orientation="h",
            marker_color=category_colors.get(category, "#98a2b3"),
            customdata=tier_cat_lines.loc[mix_tiers, category],
            hovertemplate=f"<b>%{{y}}</b> - {category}<br>Share of tier orders: %{{x:.1%}}<br>Order lines: %{{customdata:,.0f}}<extra></extra>",
        )
    )
apply_dashboard_layout(fig, "What each value tier buys (by order lines)", height=520, showlegend=True)
fig.update_layout(barmode="stack")
fig.update_xaxes(title_text="Share of tier orders", tickformat=".0%", range=[0, 1])
fig.update_yaxes(title_text="Customer value tier", automargin=True)
fig.show()

**Idea behind the design choice:** This section profiles who the customers are and how concentrated their value is. The revenue-vs-activity scatter and the Pareto curve test the same idea from two angles, that a small share of customers drives most of the revenue. The box plot and the donut split revenue across value tiers, the occupation heatmap and the demographic bar describe the segments behind those tiers. The "what each value tier buys" is supposed to describe the needs of each group.

**Possible insights:** Revenue is highly concentrated. Higher value tiers skew toward specific occupations and buy a broader mix of categories, while lower tiers are more single-category.

**Usage of interactions:** Hover reveals exact revenue, tier and segment figures, the heatmap and Pareto tooltips give precise shares, and legend toggling on the stacked and tier charts allows to isolate individual categories or tiers.

## Sales

In [37]:
# Sales and profit trend
monthly_sales = (
    trend_sales.groupby("month", as_index=False)
    .agg(revenue=("revenue", "sum"), profit=("profit", "sum"))
    .sort_values("month")
)
monthly_sales["margin"] = monthly_sales["profit"] / monthly_sales["revenue"]
fig = go.Figure()
fig.add_trace(go.Scatter(x=monthly_sales["month"], y=monthly_sales["revenue"], mode="lines+markers", name="Revenue", line=dict(color="#1769aa", width=3), hovertemplate="%{x|%b %Y}<br>Revenue: $%{y:,.0f}<extra></extra>"))
fig.add_trace(go.Scatter(x=monthly_sales["month"], y=monthly_sales["profit"], mode="lines+markers", name="Profit", line=dict(color="#2e7d32", width=3), hovertemplate="%{x|%b %Y}<br>Profit: $%{y:,.0f}<extra></extra>"))
fig.add_trace(go.Scatter(x=monthly_sales["month"], y=monthly_sales["margin"], mode="lines", name="Gross margin", yaxis="y2", line=dict(color="#f79009", width=2, dash="dot"), hovertemplate="%{x|%b %Y}<br>Gross margin: %{y:.1%}<extra></extra>"))
apply_dashboard_layout(fig, "Sales and profit trend", height=500, showlegend=True)
fig.update_yaxes(title_text="Revenue / profit", tickprefix="$", separatethousands=True)
fig.update_layout(yaxis2=dict(title="Gross margin", overlaying="y", side="right", tickformat=".0%", rangemode="tozero", showgrid=False), hovermode="x unified")
fig.show()

# Category mix over time
mix = sales_rows.groupby(["year", "category"], as_index=False)["revenue"].sum()
fig = go.Figure()
for category in sorted(mix["category"].unique()):
    rows = mix[mix["category"] == category]
    fig.add_trace(go.Bar(x=rows["year"], y=rows["revenue"], name=category, marker_color=category_colors.get(category, "#98a2b3"), hovertemplate=f"%{{x}} - {category}<br>Revenue: $%{{y:,.0f}}<extra></extra>"))
apply_dashboard_layout(fig, "Category mix over time", height=460, showlegend=True)
fig.update_layout(barmode="stack")
fig.update_xaxes(title_text="Year", dtick=1)
fig.update_yaxes(title_text="Revenue", tickprefix="$", separatethousands=True)
fig.show()

# Top products
top_products = sales_rows.groupby("product", as_index=False)["revenue"].sum().sort_values("revenue", ascending=False).head(18).sort_values("revenue")
fig = revenue_bar(top_products, "revenue", "product", "Top products", orientation="h", color="#2e7d32", height=620)
fig.show()

**Idea behind the design choice:** Sales section looks at trajectory and composition. The trend chart adds gross margin on a secondary axis so we can tell whether growth is actually profitable or just bigger, the category-mix-over-time chart shows whether the product balance is shifting from year to year, and the top-products bar names the specific models that carry the business.

**Possible insights:** Revenue grows while gross margin stays roughly stable, so scaling up is not removing profitability. The category mix stays bike-dominated across the years, and a short list of bike models makes up the top products.

**Usage of interactions:** Unified hover gives revenue, profit and margin per period, legend toggles isolate categories in the mix chart, and bar hover gives the exact revenue per product.

## Stock

In [38]:
# Stock value by location
location_agg = (
    stock_rows.groupby("location", as_index=False)
    .agg(value=("value", "sum"), qty=("qty", "sum"), products=("product", "nunique"))
    .sort_values("value", ascending=False)
    .reset_index(drop=True)
)
location_agg["valuePerUnit"] = location_agg["value"] / location_agg["qty"].replace(0, np.nan)
location_agg["cumulativeShare"] = location_agg["value"].cumsum() / location_agg["value"].sum()
fig = go.Figure()
fig.add_trace(go.Bar(x=location_agg["location"], y=location_agg["value"], name="Stock value", marker_color="#7a5af8", customdata=np.stack([location_agg["qty"], location_agg["valuePerUnit"], location_agg["products"]], axis=-1), hovertemplate="%{x}<br>Stock value: $%{y:,.0f}<br>Quantity: %{customdata[0]:,.0f}<br>Value/unit: $%{customdata[1]:,.0f}<br>Products: %{customdata[2]:,.0f}<extra></extra>"))
fig.add_trace(go.Scatter(x=location_agg["location"], y=location_agg["cumulativeShare"], mode="lines+markers", name="Cumulative value share", yaxis="y2", line=dict(color="#2e7d32", width=3), hovertemplate="%{x}<br>Cumulative value share: %{y:.1%}<extra></extra>"))
apply_dashboard_layout(fig, "Stock value by location", height=560, showlegend=True)
fig.update_xaxes(title_text="Stock location", tickangle=-35)
fig.update_yaxes(title_text="Stock value", tickprefix="$", separatethousands=True)
fig.update_layout(yaxis2=dict(title="Cumulative share", overlaying="y", side="right", tickformat=".0%", range=[0, 1.05], showgrid=False), hovermode="x unified")
fig.show()

# Restock priority by coverage
priority = demand_rows[demand_rows["recommendation"] == "Restock priority"].sort_values("stockToDemand", ascending=True)
priority_chart = priority.head(15).sort_values("stockToDemand", ascending=False).copy()
priority_chart["coverLabel"] = priority_chart["daysOfCover"].map(lambda value: "n/a" if pd.isna(value) else f"{value:,.0f} d")
fig = go.Figure(go.Bar(x=priority_chart["stockToDemand"], y=priority_chart["product"], orientation="h", marker_color="#ba1a1a", customdata=np.stack([priority_chart["demand"], priority_chart["recentDemand"], priority_chart["stockQty"], priority_chart["revenue"], priority_chart["coverLabel"]], axis=-1), hovertemplate="<b>%{y}</b><br>Stock/demand: %{x:.2f}<br>Demand: %{customdata[0]:,.0f}<br>Recent demand: %{customdata[1]:,.0f}<br>Stock: %{customdata[2]:,.0f}<br>Days of cover: %{customdata[4]}<br>Revenue: $%{customdata[3]:,.0f}<extra></extra>"))
apply_dashboard_layout(fig, "Restock priority by coverage", height=620)
fig.update_xaxes(title_text="Stock / demand ratio", separatethousands=True)
fig.update_yaxes(automargin=True)
fig.show()

**Idea behind the design choice:** The stock views separated into two different questions: "where inventory value is located" and "which products are lacking relative to how fast they sell".

**Possible insights:** Stock value is concentrated in a few locations. The coverage ranking surfaces the products whose stock is low relative to their demand, which are the sensible restock candidates.

**Usage of interactions:** Hover gives the exact stock value, cumulative share and stock/demand ratio for each location and product.